# OPTIMIZE

In [56]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## Set autoCompaction and optimizeWrite to False
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "false")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "false")

try:
    print(spark.conf.get("spark.databricks.delta.optimizeWrite.enabled"))
except Exception as e:
    print("optimizeWrite not set:", e)

try:
    print(spark.conf.get("spark.databricks.delta.autoCompact.enabled"))
except Exception as e:
    print("autoCompact not set:", e)

In [57]:
src_file = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_201_99457.parquet"

df = spark.read.format('parquet').load(src_file)

df.count()

In [58]:
df.select('category').distinct().count()

In [61]:
target_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/optimize/small_files/"
df.repartition(5).write.format('delta').mode('overwrite').partitionBy('category').save(target_path)

## Small File Problem

# The Small File Problem — Full Summary

## Why is a *single* giant file also a problem?

Before covering small files, it's worth noting the opposite extreme is bad too:
forcing everything into **one file** means one task, on one executor, must hold
the *entire* dataset in memory before writing it out.

- **No parallelism** — only one core does the writing while every other executor sits idle.
- **OOM risk** — for large datasets, that one task can run out of memory trying
  to buffer everything before the write completes.

So the goal was never "fewest files possible" — it's **appropriately sized**
files. Delta's own default bin-packing target is **~1 GB per file**, a number
tuned from years of real-world Spark workload testing.

---

## Why are *too many small* files a problem?

Analogy: reading a 300-page novel that's been saved as 300 separate single-page
PDFs instead of one file. For every page, you have to *find* the right file,
*open* it, read it, then *close* it — tiny overhead repeated 300 times instead
of paid once.

<img src='https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/Small%20File%20Problem.png?raw=true' height=600/>

Same thing happens with Spark/Delta:
- Every small file requires its own **open → read → close** cycle plus a
  **metadata lookup** in the transaction log.
- Multiply this by thousands of tiny files, and you get **wasted compute and
  poor I/O** — even though the actual *data volume* hasn't changed at all.

**Real trace confirming this** (from the history log I formatted earlier):
a table had **2,400 files** for just ~9 MB of data — averaging ~4 KB per
file. All fragmentation overhead, zero benefit.

---

## Root Causes

| # | Cause | Type |
|---|---|---|
| 1 | **Over-repartitioning** — `repartition(N)` set far higher than the data volume justifies | Technical, user-controlled |
| 2 | **Partitioning on a high-cardinality column** — too many distinct values, each getting its own tiny folder | Technical, user-controlled |
| 3 | **Frequent small incremental writes** — e.g. data arriving every few minutes (streaming/near-real-time) | Business-driven, not just a config fix |

Causes #1 and #2 can be fixed **at the source** — pick a sane repartition
count, avoid high-cardinality partition columns. Cause #3 can't be "fixed"
upstream (the business genuinely needs fresh data often) — it has to be
handled with **after-the-fact or write-time compaction** instead.

---

## How Delta Solves It — Three Approaches

| Approach | When it runs | Mechanism | Trade-off |
|---|---|---|---|
| **Manual `OPTIMIZE`** (scheduled) | Triggered on demand or via a periodic job (hourly/daily) | Bin-packing: sorts files by size, packs them into ~1 GB bins | Doesn't fix anything until you run it — a batch cleanup, not prevention |
| **Optimized Write** | Automatically, on *every* write | Adds a shuffle step *before* writing so each partition gets one (or a few, if large) appropriately-sized file(s) from the start — instead of many scattered small files from independent executors | Shuffle adds **write latency** on every single write |
| **Auto Compaction** | Automatically, right after a write commits | Post-commit hook — checks if file count crosses a threshold (`autoCompact.minNumFiles`, default 50), then runs a mini-compaction | Only a safety net; does nothing if Optimized Write already prevented fragmentation |

### The bin-packing algorithm (manual `OPTIMIZE`)
- Sorts all files by size, **largest to smallest**.
- Greedily places each file into the current "bin" (~1 GB target) if it fits;
  otherwise starts a new bin.
- Result: many scattered small files become a handful of appropriately-sized ones.

### Why Optimized Write prevents the problem in the first place
Without it, each executor's task holds a *mixed* chunk of rows spanning
multiple partition values — so many tasks each drop a small file into the
*same* partition folder independently, with no coordination. Optimized Write
adds a **shuffle** step first, routing all rows for a given partition value to
one (or a few) tasks, so each partition gets cleanly written instead of
fragmented across every executor.

---

## `OPTIMIZE` Creates New Files — It Doesn't Delete Old Ones

This is the critical link to `VACUUM` (covered in the earlier summary):

- 5 small files --OPTIMIZE--> 1 new file (old 5 tombstoned, not deleted)
- = 6 files on disk total (5 tombstoned + 1 active)
- --VACUUM--> old 5 physically removed --> 1 file remains


`OPTIMIZE` alone **does not reclaim storage** — the old fragmented files stay
on disk, just marked as no-longer-referenced (tombstoned). Only `VACUUM`
physically deletes them — and doing so with low/zero retention **breaks time
travel** to any version that depended on those files, as demonstrated in the
history trace earlier.

---

## Quick Decision Guide

| Situation | Best fix |
|---|---|
| One-off cleanup of an already-fragmented table | Manual `OPTIMIZE` |
| Want fragmentation **prevented**, willing to pay shuffle cost per write | **Optimized Write** |
| Want a safety net for writes that bypass Optimized Write (other pipelines, `MERGE`s, streaming micro-batches) | **Auto Compaction** |
| Reclaim disk space after any of the above | `VACUUM` — separately, and carefully with retention |

**One-line summary:** small files ≠ deleted by `OPTIMIZE`; `OPTIMIZE`
*replaces* them with clean files and tombstones the old ones — `VACUUM` is
the separate, deliberate step that actually frees the storage, at the cost of
losing time-travel to whatever it deletes.


In [62]:
%%time

df_src = spark.read.format('delta').load(target_path)
df_src = df_src.where(df_src.category == 'Clothing').collect()

Optimize run

In [63]:
df_delta = DeltaTable.forPath(spark, target_path)
df_delta.optimize().executeCompaction()

In [64]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [65]:
df_delta.vacuum(0)

In [66]:
%%time

df_src = spark.read.format('delta').load(target_path)
df_src = df_src.where(df_src.category == 'Clothing').collect()

## optimize with predicate

Running opitimize only in particular partition

In [68]:
df_fruits = df.filter(F.col("category")=="Clothing").withColumn("category", F.lit("Fruits"))
df_fruits.repartition(5).write.format('delta').partitionBy('category').mode('overwrite').save(target_path)  ## Simulate the small file problem

In [76]:
delta_table_fruits = DeltaTable.forPath(spark, target_path)
delta_table_fruits.optimize().where("category = 'Fruits'").executeCompaction()

In [79]:
delta_table_fruits.vacuum(0)

## Optimized Write

<img src="https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/Optimization%20Techniques.png?raw=true">

In [80]:
target_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/optimize/small_files_v2/"
df.repartition(300).write.format('delta').mode('overwrite').partitionBy('category').save(target_path)

In [81]:
%%time
df_ex2 = spark.read.format('delta').load(target_path)
df_out = df_ex2.where(df_ex2.category == "Clothing").collect()

In [85]:
target_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/optimize/small_files_v3/"
df.repartition(300).write.format('delta').partitionBy('category').mode('overwrite').option('optimizeWrite', True).save(target_path)

In [86]:
%%time
df_ex3 = spark.read.format('delta').load(target_path)
df_out = df_ex3.where(df_ex3.category == "Clothing").collect()

## Auto Compaction

Note: 
1. set optimized write to false here so that we can see the autocompaction in action
2. auto compaction would compact the 

In [97]:
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "false")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", True)

spark.conf.set("spark.databricks.delta.autoCompact.minNumFiles", "3")
spark.conf.get("spark.databricks.delta.autoCompact.minNumFiles")

In [96]:
print(spark.conf.get("spark.databricks.delta.autoCompact.enabled"))
print(spark.conf.get("spark.databricks.delta.autoCompact.minNumFiles"))

In [98]:
df_detergent = df.filter(F.col("category")=='Clothing').withColumn("category", F.lit("Detergent"))

df_detergent.repartition(10).write.mode('overwrite').partitionBy('category').save(target_path)

## VACUUM

### Delta Table History — OPTIMIZE + VACUUM Walkthrough

A real `DESCRIBE HISTORY` trace showing the full lifecycle: a small-file-heavy write,
followed by compaction, followed by physical cleanup.

| Version | Operation | Timestamp | Key Parameters | Key Metrics | What Happened |
|---|---|---|---|---|---|
| 0 | **WRITE** | 04:59:07.598 | `mode: Overwrite`<br>`partitionBy: [category]` | `numFiles: 2400`<br>`numOutputRows: 99,257`<br>`numOutputBytes: ~9.13 MB` | Initial write — badly fragmented. **2,400 files** for ~9 MB of data ≈ **~4 KB per file on average**. Classic small-file problem. |
| 1 | **OPTIMIZE** | 05:34:41.786 | `zOrderBy: []`<br>`auto: false` (manual) | `numRemovedFiles: 2400`<br>`numAddedFiles: 8`<br>`numAddedBytes: ~1.35 MB`*<br>`minFileSize: 75 KB`<br>`maxFileSize: 480 KB`<br>`p50FileSize: 145 KB` | Bin-packing compaction: **2,400 tiny files → 8 well-sized files**. Old files tombstoned (marked removed), not yet deleted. |
| 2 | **VACUUM START** | 05:35:19.120 | `retentionCheckEnabled: false`<br>`specifiedRetentionMillis: 0` | `numFilesToDelete: 2400`<br>`sizeOfDataToDelete: ~9.13 MB` | Retention safety check disabled, **0 ms retention** — forces immediate cleanup instead of the default 7-day (604,800,000 ms) wait. |
| 3 | **VACUUM END** | 05:35:52.153 | `status: COMPLETED` | `numDeletedFiles: 2400`<br>`numVacuumedDirectories: 9` | The 2,400 tombstoned files are **physically deleted** from storage across 9 partition directories. |

*\* Note: `numOutputBytes` at v0 (9,583,931) exactly equals `numRemovedBytes` at v1 and `sizeOfDataToDelete` at v2 — confirming the same physical data volume is being tracked through the whole compact → tombstone → delete lifecycle.*

---

### The story this log tells

1. **v0 — the problem gets created.** A single overwrite writes 99,257 rows into **2,400 files**, averaging ~4 KB each — a textbook small-file problem, likely from over-partitioning or over-repartitioning at write time.
2. **v1 — `OPTIMIZE` fixes the layout.** Bin-packing consolidates all 2,400 fragments into just **8 files** (75 KB–480 KB each), cutting total size from ~9.1 MB to ~1.35 MB by eliminating Parquet overhead from thousands of tiny file headers/footers. The 2,400 old files aren't deleted — just tombstoned (soft-deleted) in the log.
3. **v2 → v3 — `VACUUM` reclaims the space.** With `retentionCheckEnabled=false` and `retentionMillis=0`, the default 7-day safety window is bypassed, and all 2,400 tombstoned files are **physically removed** from 9 partition directories.

### Key takeaway
This trace is a clean real-world confirmation of the **`OPTIMIZE` → `VACUUM`** pattern:
`OPTIMIZE` creates new clean files but *leaves old ones on disk* (recoverable via time travel) →
`VACUUM` is a **separate, deliberate step** to actually reclaim that storage — and running it with `0` retention (as here) means versions 0's data is now **permanently unrecoverable** via time travel.


In [106]:
file_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/optimize/small_files_v2/"

delta_table = DeltaTable.forPath(spark, file_path)

display(delta_table.history())

In [107]:
delta_table.optimize().executeCompaction()

In [108]:
delta_table.vacuum(0)

In [111]:
display(delta_table.history().select("version", "timestamp", "operation", "operationParameters", "operationMetrics", "engineInfo"))